In [ ]:
import pandas as pd

df = pd.read_csv("../data/cleaned_cars.csv")

In [2]:
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 243784 entries, 0 to 243783
Data columns (total 13 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   brand                     243784 non-null  str    
 1   model                     243784 non-null  str    
 2   color                     243700 non-null  str    
 3   registration_date         243784 non-null  str    
 4   year                      243784 non-null  int64  
 5   price_in_euro             243784 non-null  int64  
 6   power_kw                  243657 non-null  float64
 7   power_ps                  243657 non-null  float64
 8   transmission_type         243784 non-null  str    
 9   fuel_type                 243784 non-null  str    
 10  fuel_consumption_l_100km  217234 non-null  str    
 11  fuel_consumption_g_km     243784 non-null  str    
 12  mileage_in_km             243731 non-null  float64
dtypes: float64(3), int64(2), str(8)
memory usage: 24.2 MB


,brand,model,color,registration_date,year,price_in_euro,power_kw,power_ps,transmission_type,fuel_type,fuel_consumption_l_100km,fuel_consumption_g_km,mileage_in_km
0,alfa-romeo,Alfa Romeo GTV,red,10/1995,1995,1300,148.0,201.0,Manual,Petrol,"10,9 l/100 km",260 g/km,160500.0
1,alfa-romeo,Alfa Romeo 164,black,02/1995,1995,24900,191.0,260.0,Manual,Petrol,NaN,- (g/km),190000.0
2,alfa-romeo,Alfa Romeo Spider,black,02/1995,1995,5900,110.0,150.0,Unknown,Petrol,NaN,- (g/km),129000.0
3,alfa-romeo,Alfa Romeo Spider,black,07/1995,1995,4900,110.0,150.0,Manual,Petrol,"9,5 l/100 km",225 g/km,189500.0
4,alfa-romeo,Alfa Romeo 164,red,11/1996,1996,17950,132.0,179.0,Manual,Petrol,"7,2 l/100 km",- (g/km),96127.0


In [3]:
print(df.isnull().sum())

brand                           0
model                           0
color                          84
registration_date               0
year                            0
price_in_euro                   0
power_kw                      127
power_ps                      127
transmission_type               0
fuel_type                       0
fuel_consumption_l_100km    26550
fuel_consumption_g_km           0
mileage_in_km                  53
dtype: int64


In [4]:
df_xgb = df.copy()

In [5]:
df_xgb = df_xgb.drop(columns=['registration_date'])

In [6]:
df_xgb['fuel_consumption_l_100km'].head(10)

0    10,9 l/100 km
1              NaN
2              NaN
3     9,5 l/100 km
4     7,2 l/100 km
5     9,5 l/100 km
6     8,8 l/100 km
7    13,4 l/100 km
8      11 l/100 km
9     9,2 l/100 km
Name: fuel_consumption_l_100km, dtype: str

In [7]:
df_xgb['fuel_consumption_l_100km'] = (
    df_xgb['fuel_consumption_l_100km']
    .str.extract(r'(\d+\.?\d*)')[0]
    .astype(float)
)

In [8]:
df_xgb['fuel_consumption_g_km'] = (
    df_xgb['fuel_consumption_g_km']
    .str.extract(r'(\d+\.?\d*)')[0]
    .astype(float)
)

In [9]:
df_xgb[['fuel_consumption_l_100km',
        'fuel_consumption_g_km']].info()

<class 'pandas.DataFrame'>
RangeIndex: 243784 entries, 0 to 243783
Data columns (total 2 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   fuel_consumption_l_100km  217234 non-null  float64
 1   fuel_consumption_g_km     207380 non-null  float64
dtypes: float64(2)
memory usage: 3.7 MB


In [10]:
df_xgb['power_kw'] = (
    df_xgb['power_kw']
    .fillna(df_xgb['power_kw'].median())
)

In [11]:
df_xgb['mileage_in_km'] = (
    df_xgb['mileage_in_km']
    .fillna(df_xgb['mileage_in_km'].median())
)

In [12]:
df_xgb['color'] = (
    df_xgb['color']
    .fillna('Unknown')
)

In [13]:
df_xgb['fuel_consumption_l_100km'] = (
    df_xgb['fuel_consumption_l_100km']
    .fillna(
        df_xgb['fuel_consumption_l_100km'].median()
    )
)

In [14]:
df_xgb = df_xgb.drop(columns=['power_ps'])

In [15]:
import numpy as np

df_xgb['price_log'] = np.log(df_xgb['price_in_euro'])

In [16]:
df_xgb.info()

<class 'pandas.DataFrame'>
RangeIndex: 243784 entries, 0 to 243783
Data columns (total 12 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   brand                     243784 non-null  str    
 1   model                     243784 non-null  str    
 2   color                     243784 non-null  str    
 3   year                      243784 non-null  int64  
 4   price_in_euro             243784 non-null  int64  
 5   power_kw                  243784 non-null  float64
 6   transmission_type         243784 non-null  str    
 7   fuel_type                 243784 non-null  str    
 8   fuel_consumption_l_100km  243784 non-null  float64
 9   fuel_consumption_g_km     207380 non-null  float64
 10  mileage_in_km             243784 non-null  float64
 11  price_log                 243784 non-null  float64
dtypes: float64(5), int64(2), str(5)
memory usage: 22.3 MB


In [17]:
df['price_log'] = np.log(df['price_in_euro'])

In [18]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['price_in_euro', 'price_log'])
y = df['price_log']


X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [19]:
X_train['model'].nunique(), X_test['model'].nunique()

(1276, 1087)

In [20]:
model_target = pd.DataFrame({
    'model': X_train['model'],
    'target': y_train
})

In [21]:
model_mean = (
    model_target
    .groupby('model')['target']
    .mean()
)

In [22]:
model_mean.head()

model
Alfa Romeo        8.608277
Alfa Romeo 145    8.183227
Alfa Romeo 146    7.785433
Alfa Romeo 147    7.960060
Alfa Romeo 155    8.980927
Name: target, dtype: float64

In [23]:
X_train['model_target'] = X_train['model'].map(model_mean)

In [24]:
X_test['model_target'] = X_test['model'].map(model_mean)

In [25]:
X_test['model_target'].isna().sum()

40

In [26]:
global_mean = y_train.mean()

X_train['model_target'] = (
    X_train['model_target']
    .fillna(global_mean)
)

X_test['model_target'] = (
    X_test['model_target']
    .fillna(global_mean)
)

In [27]:
X_train = X_train.drop(columns=['model'])
X_test = X_test.drop(columns=['model'])

In [28]:
X_train.dtypes

brand                           str
color                           str
registration_date               str
year                          int64
power_kw                    float64
power_ps                    float64
transmission_type               str
fuel_type                       str
fuel_consumption_l_100km        str
fuel_consumption_g_km           str
mileage_in_km               float64
model_target                float64
dtype: object

In [29]:
X_train = X_train.drop(columns=['registration_date'])
X_test = X_test.drop(columns=['registration_date'])

In [30]:
for data in [X_train, X_test]:
    data['fuel_consumption_l_100km'] = (
        data['fuel_consumption_l_100km']
        .str.extract(r'(\d+\.?\d*)')[0]
        .astype(float)
    )

In [31]:
for data in [X_train, X_test]:
    data['fuel_consumption_g_km'] = (
        data['fuel_consumption_g_km']
        .str.extract(r'(\d+\.?\d*)')[0]
        .astype(float)
    )

In [32]:
X_train[['fuel_consumption_l_100km',
         'fuel_consumption_g_km']].head()

,fuel_consumption_l_100km,fuel_consumption_g_km
177554,5.0,116.0
30092,5.0,NaN
123164,6.0,136.0
115853,11.0,278.0
151544,4.0,99.0


In [33]:
for col in ['fuel_consumption_l_100km',
            'fuel_consumption_g_km']:

    median_value = X_train[col].median()

    X_train[col] = X_train[col].fillna(median_value)
    X_test[col] = X_test[col].fillna(median_value)

In [34]:
X_train = X_train.drop(columns=['power_ps'])
X_test = X_test.drop(columns=['power_ps'])

In [35]:
categorical_cols = [
    'brand',
    'color',
    'transmission_type',
    'fuel_type'
]

In [36]:
X_train = pd.get_dummies(
    X_train,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

X_test = pd.get_dummies(
    X_test,
    columns=categorical_cols,
    drop_first=True,
    dtype=int
)

In [37]:
X_train, X_test = X_train.align(
    X_test,
    join='left',
    axis=1,
    fill_value=0
)

In [38]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 195027 entries, 177554 to 121958
Data columns (total 78 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   year                              195027 non-null  int64  
 1   power_kw                          194931 non-null  float64
 2   fuel_consumption_l_100km          195027 non-null  float64
 3   fuel_consumption_g_km             195027 non-null  float64
 4   mileage_in_km                     194990 non-null  float64
 5   model_target                      195027 non-null  float64
 6   brand_aston-martin                195027 non-null  int32  
 7   brand_audi                        195027 non-null  int32  
 8   brand_bentley                     195027 non-null  int32  
 9   brand_bmw                         195027 non-null  int32  
 10  brand_cadillac                    195027 non-null  int32  
 11  brand_chevrolet                   195027 non-null  int32  
 12 

In [39]:
from xgboost import XGBRegressor

In [40]:
X_train.info()

<class 'pandas.DataFrame'>
Index: 195027 entries, 177554 to 121958
Data columns (total 78 columns):
 #   Column                            Non-Null Count   Dtype  
---  ------                            --------------   -----  
 0   year                              195027 non-null  int64  
 1   power_kw                          194931 non-null  float64
 2   fuel_consumption_l_100km          195027 non-null  float64
 3   fuel_consumption_g_km             195027 non-null  float64
 4   mileage_in_km                     194990 non-null  float64
 5   model_target                      195027 non-null  float64
 6   brand_aston-martin                195027 non-null  int32  
 7   brand_audi                        195027 non-null  int32  
 8   brand_bentley                     195027 non-null  int32  
 9   brand_bmw                         195027 non-null  int32  
 10  brand_cadillac                    195027 non-null  int32  
 11  brand_chevrolet                   195027 non-null  int32  
 12 

In [41]:
xgb_model = XGBRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='reg:squarederror',
    random_state=42,
    n_jobs=-1
)

In [46]:
xgb_model.fit(
    X_train,
    y_train
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [47]:
pred_log = xgb_model.predict(X_test)

In [48]:
import numpy as np

pred_price = np.exp(pred_log)
true_price = np.exp(y_test)

In [49]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


mae = mean_absolute_error(
    true_price,
    pred_price
)

rmse = np.sqrt(
    mean_squared_error(
        true_price,
        pred_price
    )
)

r2 = r2_score(
    true_price,
    pred_price
)


print(f"MAE: {mae:.2f} €")
print(f"RMSE: {rmse:.2f} €")
print(f"R²: {r2:.4f}")

MAE: 3434.08 €
RMSE: 20626.40 €
R²: 0.7198


In [50]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb_model.feature_importances_
})

importance = importance.sort_values(
    'importance',
    ascending=False
)

importance.head(20)

,feature,importance
0,year,0.163745
65,transmission_type_Manual,0.115897
5,model_target,0.098078
1,power_kw,0.070903
18,brand_ferrari,0.059195
40,brand_porsche,0.054628
4,mileage_in_km,0.025951
68,fuel_type_Diesel,0.025794
50,brand_volkswagen,0.022062
34,brand_mercedes-benz,0.021938
